In [1]:
import os
import importlib
os.environ["CUDA_VISIBLE_DEVICES"]="1,2"
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM, AutoModel
from datasets import load_dataset
import torch
#from sentence_transformers import SentenceTransformer, InputExample, losses
#from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction
from torch.utils.data import DataLoader
from datasets import Dataset
import pandas as pd
from collections import defaultdict
import re
import numpy as np
import time

device1 = 'cuda:0'
device2 = 'cuda:1'
data_dir = '/raid/deallab/SF_RAG_Data/ASQA'
# data_dir = '../data'

/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#load embeddings
embedd_test_path = f'{data_dir}/test/embedd_test.npy'
evidence_embeddings = np.load(embedd_test_path)
print(evidence_embeddings.shape)
evidence_embeddings = torch.from_numpy(evidence_embeddings).to(device1)

#load evidence
evidence_test_path = f'{data_dir}/test/evidence_test.csv'
evidence_df = pd.read_csv(evidence_test_path)

#load qa data
qa_df=pd.read_csv(f'{data_dir}/test/qa_test.csv') #data=df[['question','long_answers']] # questions=data['question'] #references = [row.to_dict() for i, row in df.iterrows() if i < len(questions)]
qa_df.head()

(21586, 4096)


,id,sample_id,question,follow_up_questions,long_answers,short_answers
0,c2687961-0957-45cb-bae0-42314e38f790,-7013890438520559398,Who has the highest goals in world football?,"[""Who has the highest goals in men's world int...","[""Ali Dael has the highest goals in men's worl...","[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'..."
1,26830122-8240-40a9-aaff-d9731d53b197,7089015503030534342,Who is the original artist of sound of silence?,['Who is the original artist of sound of silen...,[' The original artist of the song sound of si...,"[['Simon & Garfunkel', 'Paul Simon and Art Gar..."
2,268116a9-5ecb-4364-8da4-4a648f9d5b43,8793099883447006698,When was the first apple i phone made?,"['When was the first apple i phone released?',...",['The iPhone beta was created in 2004 to test ...,"[['June 29, 2007'], ['2004'], ['June 29, 2007...."
3,efb4810e-637b-4954-a776-3c2d05d1290c,-881464876144297194,Who played the weasley brothers in harry potter?,['Who played Bill weasley in Harry Potter and...,['Rupert Grint played Ron Weasley in all the H...,"[['Richard Fish'], ['Chris Rankin'], ['James P..."
4,99817eba-d32a-4c4d-9fe2-93a50ae1d367,1650309494326541834,How many state parks are there in virginia?,['How many state parks are there in virginia i...,['When the Virginia state park system was form...,"[['six'], ['38'], ['6'], ['38']]"


In [3]:
#load quantized model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_storage=torch.bfloat16,
)

# load model with tokenizer
model = AutoModel.from_pretrained(
    'nvidia/NV-Embed-v2', 
    trust_remote_code=True,
    quantization_config = bnb_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage =True,
)
model.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.67s/it]


NVEmbedModel(
  (latent_attention_model): LatentAttentionModel(
    (cross_attend_blocks): ModuleList(
      (0): PreNorm(
        (fn): Attention(
          (to_q): Linear4bit(in_features=4096, out_features=32768, bias=False)
          (to_kv): Linear4bit(in_features=4096, out_features=65536, bias=False)
          (to_out): Linear4bit(in_features=32768, out_features=4096, bias=False)
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
        (norm_context): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
      (1): PreNorm(
        (fn): FeedForward(
          (net): Sequential(
            (0): Linear4bit(in_features=4096, out_features=32768, bias=True)
            (1): GEGLU()
            (2): Linear4bit(in_features=16384, out_features=4096, bias=True)
          )
        )
        (norm): LayerNorm((4096,), eps=1e-05, elementwise_affine=True)
      )
    )
  )
  (embedding_model): BidirectionalMistralModel(
    (embed_tokens): Embedding(

In [4]:
#load tokenizer
tokenizer_gen = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3.1-8B-Instruct")
# cache_dir= '/raid/deallab/.cache')
tokenizer_gen.pad_token = tokenizer_gen.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    # bnb_4bit_quant_type="nf4",
    # bnb_4bit_compute_dtype=torch.bfloat16,
    # bnb_4bit_use_double_quant=True,
    # bnb_4bit_quant_storage=torch.bfloat16,
)

model_gen = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map= 'auto',
    # cache_dir= '/raid/deallab/.cache'
)
model_gen.eval()

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.38it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): Ll

In [5]:
import pandas as pd
evidence_test_path = f'/raid/deallab/SF_RAG_Data/ASQA/test/evidence_test.csv'

evidence_text = pd.read_csv(evidence_test_path)

In [6]:
evidence_text

,text
0,Document: International Federation of Football...
1,Document: International Federation of Football...
2,Document: International Federation of Football...
3,Document: International Federation of Football...
4,Document: International Federation of Football...
...,...
21581,Document: Sign of the Times (Harry Styles song...
21582,Document: Sign of the Times (Harry Styles song...
21583,Document: Sign of the Times (Harry Styles song...
21584,Document: Sign of the Times (Harry Styles song...


In [8]:
evidence_text_list = evidence_text['text'].tolist()

In [22]:
evidence_text_list[0]

"Document: International Federation of Football History & Statistics\n\n\n\nTitle: N/A\nDescription: N/A\n\nFields:\nFormation: 1984\nHeadquarters: Lausanne, Switzerland\nOfficial language: English, French, Spanish, German\nPresident: Saleh Bahwini[1]\nWebsite: http://iffhs.de/\n\nSingles Chronology:\n\nExternal Links:\nThe International Federation of Football History & Statistics (IFFHS) is an organization that chronicles the history and records of association football.[2][3][4] \nIt was founded on 27 March 1984 in Leipzig by Alfredo Pöge with the blessings of general secretary of the FIFA at the time, Helmut Käser.[2] The IFFHS was based at Al-Muroor Street 147, Abu Dhabi for some time but, in 2010, relocated to Bonn, Germany.[5]\n\nDuring its early stages, and until 2002, the IFFHS concentrated on publishing the quarterly magazines Fußball-Weltzeitschrift, Libero spezial deutsch and Libero international.[6] When these had to be discontinued for reasons which were not officially told

In [57]:
from langchain_text_splitters import TokenTextSplitter

text_splitter = TokenTextSplitter(
    chunk_size=500,  # 청크 크기를 10으로 설정합니다.
    chunk_overlap=50,  # 청크 간 중복을 0으로 설정합니다.
)
# combined_text = " ".join(evidence_text_list)
# texts = text_splitter.split_text(combined_text)
split_texts = [text_splitter.split_text(text)[0] for text in evidence_text_list]
print(split_texts[0])

Document: International Federation of Football History & Statistics



Title: N/A
Description: N/A

Fields:
Formation: 1984
Headquarters: Lausanne, Switzerland
Official language: English, French, Spanish, German
President: Saleh Bahwini[1]
Website: http://iffhs.de/

Singles Chronology:

External Links:
The International Federation of Football History & Statistics (IFFHS) is an organization that chronicles the history and records of association football.[2][3][4] 
It was founded on 27 March 1984 in Leipzig by Alfredo Pöge with the blessings of general secretary of the FIFA at the time, Helmut Käser.[2] The IFFHS was based at Al-Muroor Street 147, Abu Dhabi for some time but, in 2010, relocated to Bonn, Germany.[5]

During its early stages, and until 2002, the IFFHS concentrated on publishing the quarterly magazines Fußball-Weltzeitschrift, Libero spezial deutsch and Libero international.[6] When these had to be discontinued for reasons which were not officially told, the organization pu

In [72]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.vectorstores import FAISS

# bm25 retriever와 faiss retriever를 초기화합니다.
bm25_retriever = BM25Retriever.from_texts(
    evidence_text_list,
)
bm25_retriever.k = 10  # BM25Retriever의 검색 결과 개수를 1로 설정합니다.

# embedding = model
# faiss_vectorstore = FAISS.from_texts(
#     evidence_text,
#     embedding,
# )
# faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k": 2})

# # 앙상블 retriever를 초기화합니다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, faiss_retriever],
#     weights=[0.7, 0.3],
# )

In [75]:
from langchain_community.document_transformers import LongContextReorder

def bm25_retrieve(query):
    bm25_result = bm25_retriever.invoke(query)
    bm25_docs=list()

    print("[BM25 Retriever]")
    for doc in bm25_result:
        # print(f"Content: {doc.page_content}")
        # print()
        bm25_docs.append(doc.page_content)
    reordering = LongContextReorder()
    bm25_docs = reordering.transform_documents(bm25_docs)
    return bm25_docs

In [76]:
res=bm25_retrieve("Who has the highest goals in world football?")
res

[BM25 Retriever]


['Document: List of players with five or more goals in an NHL game\n\nThis is a list of players who have scored five or more goals in a National Hockey League (NHL) game. Scoring five or more goals in a single game is considered a great feat, as it has only been accomplished 61 times, by 45 players, in the history of the league.[1] The first player to do so was Joe Malone, with the Montreal Canadiens, in the first ever NHL game, on December 19, 1917. The most recent player to do so was Patrik Laine, with the Winnipeg Jets, in the 101st NHL season of play, on November 24, 2018.\n\nIn addition to being first, Joe Malone holds the overall record with five different five-or-more goal games, including the NHL record seven goals in a game, as well as a six-goal game and three five-goal games – all in the first three seasons of the NHL’s existence. He is also the only player to record a five-goal game with more than one team, accomplishing his first three with the Montreal Canadiens and his l

In [5]:
# retrive docs from the document embeddings
def retrieve_documents(query):
    max_length = 1024
    
    #query prefix
    task_name_to_instruct = {"example": "Given a question, retrieve passages that answer the question",}
    query_prefix = "Instruct: "+task_name_to_instruct["example"]+"\nQuery: "
    
    query_embedding = model.encode([query],instruction=query_prefix, max_length=max_length).to(device1)
    
    similarities = torch.nn.functional.cosine_similarity(query_embedding, evidence_embeddings)

    top_results = similarities.argsort(descending=True)[:10].cpu().detach().numpy()
    res=[evidence_df.loc[idx, 'text'] for idx in top_results if idx < len(evidence_df)]
        
    return res

In [6]:
def summarize(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [43]:
def HyDE(query, docs):
    prompt = """
    In a Retrieval Augmentation Generation system, documents close to the query vector are as follows:
    ---------------------
    {0}
    ---------------------
    Identify entities and contexts in a query, and use them to extract and summarize only relevant content from documents.
    Query: {1}
    Answer:
    """.format('\n'.join(docs), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device2)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device2)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [7]:
def answer(query, context):
    prompt = """
    Context information is below.
    ---------------------
    {0}
    ---------------------
    Given the context information and not prior knowledge, answer the query.
    Query: {1}
    Answer:
    """.format('\n'.join(context), query)
    input_ids = tokenizer_gen.apply_chat_template([{"role":'user', "content":prompt}], return_tensors='pt').to(device1)

    attention_mask = (input_ids != tokenizer_gen.pad_token_id).long().to(device1)

    out = model_gen.generate(input_ids, attention_mask=attention_mask, pad_token_id=tokenizer_gen.pad_token_id, max_new_tokens = 512)
    res = tokenizer_gen.decode(out[0]).split('<|end_header_id|>')[-1] 
    return [re.sub('\n|<\|eot_id\|>', '', res)]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3')
model.eval()

with torch.no_grad():
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    scores = exp_normalize(scores.numpy()) 
    
print(np.round(scores * 100, 2))

In [77]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    # retrieved_docs = retrieve_documents(query)
    bm25_docs = bm25_retrieve(query)
    # summary=summarize(query,retrieved_docs)
    # print(summary)
    ans=answer(query,bm25_docs)
    print('Final ans:', ans)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]

[BM25 Retriever]
Final ans: ['According to the given context information, the player with the highest goals in world football is Ali Daei of Iran, who has scored 109 goals in international football.']
According to the given context information, the player with the highest goals in world football is Ali Daei of Iran, who has scored 109 goals in international football.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:04<01:20,  4.26s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.8047415018081665, 'start': 99, 'end': 107, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.007298500742763281, 'start': 99, 'end': 107, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.010482086800038815, 'start': 99, 'end': 107, 'answer': 'Ali Daei'}
{'rougeLsum': 38.235294117647065, 'length': 28.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
[BM25 Retriever]
Final ans: ['The query is about the song "The Sound of Silence" and its original artist. However, the provided context information does not contain any information about the song or its artist.But, according to general knowledge, the song "The Sound of Silence" is by t

 10%|█         | 2/20 [00:10<01:38,  5.46s/it]

...

 95%|█████████▌| 19/20 [01:39<00:05,  5.44s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.055483803153038025, 'start': 221, 'end': 233, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.012239625677466393, 'start': 221, 'end': 233, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.6880030632019043, 'start': 221, 'end': 233, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 1.9201946997782215e-05, 'start': 221, 'end': 233, 'answer': 'Heath Ledger'}
{'rougeLsum': 40.0, 'length': 49.0, 'str_em': 50.0, 'Disambig-F1': 50.0}
[BM25 Retriever]
Final ans: ['Yes,

100%|██████████| 20/20 [01:42<00:00,  5.12s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.2116268426179886, 'start': 56, 'end': 64, 'answer': 'software'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.011645897291600704, 'start': 49, 'end': 64, 'answer': 'a free software'}
{'rougeLsum': 40.81632653061225, 'length': 12.0, 'str_em': 0.0, 'Disambig-F1': 20.0}


rougeLsum      26.991286
length         44.650000
str_em         28.750000
Disambig-F1    24.097222
dtype: float64

# Baseline

In [71]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    ans=answer(query,retrieved_docs)
    print('Final ans:', ans)
    scores=evaluate(ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Final ans: ['Based on the provided information, the player with the highest goals in world football is Ali Daei from Iran, with a total of 109 international goals.']
Based on the provided information, the player with the highest goals in world football is Ali Daei from Iran, with a total of 109 international goals.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:04<01:18,  4.11s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.9068320989608765, 'start': 90, 'end': 98, 'answer': 'Ali Daei'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.17590554058551788, 'start': 90, 'end': 98, 'answer': 'Ali Daei'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 0.03374573215842247, 'start': 90, 'end': 98, 'answer': 'Ali Daei'}
{'rougeLsum': 36.36363636363637, 'length': 26.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
Final ans: ['Simon & Garfunkel.']
Simon & Garfunkel.
Who is the original artist of sound of silence?
['Who is the original artist of sound of silence, the song, released in 1964?', 'Who is the original artist of sound of silence, the album?', 'Who is the original artist of sound of silence, 

 10%|█         | 2/20 [00:06<00:56,  3.13s/it]

...

 95%|█████████▌| 19/20 [01:17<00:05,  5.70s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9766072630882263, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 3.505659333313815e-05, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9249846935272217, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.0006961016915738583, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
{'rougeLsum': 36.36363636363637, 'length': 15.0, 'str_em': 50.0, 'Disambig-F1': 50.0}
Final ans: ['Yes, Microsoft Live Mo

100%|██████████| 20/20 [01:21<00:00,  4.08s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.7824744582176208, 'start': 125, 'end': 133, 'answer': 'Freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.4667998254299164, 'start': 125, 'end': 133, 'answer': 'Freeware'}
{'rougeLsum': 45.90163934426229, 'length': 26.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      30.726452
length         27.300000
str_em         43.333333
Disambig-F1    44.180556
dtype: float64

In [9]:
from tqdm import tqdm
from evaluation import evaluate

stop_iteration = 20

scores_list=[]
for idx, row in tqdm(qa_df.iterrows(), total=min([stop_iteration, len(qa_df)])):
    if idx == stop_iteration: break
    query = row['question']
    retrieved_docs = retrieve_documents(query)
    ans=answer(query,retrieved_docs)
    print('First ans:', ans[0])
    ans_docs=retrieve_documents(ans[0])
    final_ans=answer(query,ans_docs)
    print('Second ans:', final_ans[0])
    scores=evaluate(final_ans, [row.to_dict()])
    scores_list.append(scores)
    scores_df=pd.DataFrame(scores_list)
    print(scores)
        
scores_df=pd.DataFrame(scores_list)
scores_df.mean()

  0%|          | 0/20 [00:00<?, ?it/s]/home/dataconv/.cache/huggingface/modules/transformers_modules/nvidia/NV-Embed-v2/5130cf1daf847c1bacee854a6ef1ca939e747fb2/modeling_nvembed.py:349: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  'input_ids': torch.tensor(batch_dict.get('input_ids').to(batch_dict.get('input_ids')).long()),
/home/dataconv/anaconda3/envs/sf_rag_djk/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


First ans: Based on the provided context information, the answer to the query "Who has the highest goals in world football?" is Ali Daei of Iran, with 109 goals in international football.
Second ans: Josef Bican with 805 goals is the player with the highest goals in world football, however, his total goals are not from international football but from all levels of football.
Josef Bican with 805 goals is the player with the highest goals in world football, however, his total goals are not from international football but from all levels of football.
Who has the highest goals in world football?
["Who has the highest goals in men's world international football?", "Who has the highest goals all-time in men's football?", "Who has the highest goals in women's world international football?"]
[['Daei', 'Ali Daei'], ['Bican', 'Josef Bican'], ['Sinclair', 'Christine Sinclair']]


  5%|▌         | 1/20 [00:08<02:41,  8.49s/it]

follow question : Who has the highest goals in men's world international football?
short answer : ['Daei', 'Ali Daei']
{'score': 0.9422071576118469, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals all-time in men's football?
short answer : ['Bican', 'Josef Bican']
{'score': 0.06932494789361954, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
follow question : Who has the highest goals in women's world international football?
short answer : ['Sinclair', 'Christine Sinclair']
{'score': 4.98272966069635e-05, 'start': 0, 'end': 11, 'answer': 'Josef Bican'}
{'rougeLsum': 40.0, 'length': 30.0, 'str_em': 33.33333333333333, 'Disambig-F1': 33.33333333333333}
First ans: The original artist of "The Sound of Silence" is Simon & Garfunkel, a duo composed of Paul Simon and Art Garfunkel.
Second ans: The original artist of "The Sound of Silence" is Simon & Garfunkel, a duo consisting of Paul Simon and Art Garfunkel.
The original artist of "The Sound of Sile

 10%|█         | 2/20 [00:15<02:14,  7.48s/it]

...

 95%|█████████▌| 19/20 [02:38<00:09,  9.34s/it]

follow question : Who plays patrick in  the 1999 film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger', 'Ledger']
{'score': 0.9754783511161804, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the 2009 tv series 10 things i hate about you?
short answer : ['Ethan Peck', 'Peck', 'Ethan Gregory Peck']
{'score': 0.8937796354293823, 'start': 85, 'end': 95, 'answer': 'Ethan Peck'}
follow question : Who plays patrick in the film 10 things i hate about you?
short answer : ['Heath Andrew Ledger', 'Heath Ledger']
{'score': 0.9013998508453369, 'start': 0, 'end': 12, 'answer': 'Heath Ledger'}
follow question : Who plays patrick in the TV series 10 things i hate about you?
short answer : ['Ethan Peck', 'Ethan Gregory Peck']
{'score': 0.9357442259788513, 'start': 85, 'end': 95, 'answer': 'Ethan Peck'}
{'rougeLsum': 48.78048780487806, 'length': 32.0, 'str_em': 100.0, 'Disambig-F1': 100.0}
First ans: Yes, Microsoft Live Movie Make

100%|██████████| 20/20 [02:44<00:00,  8.23s/it]

follow question : Microsoft live movie maker is an example of a freely licensed software, often called free what?
short answer : ['freeware']
{'score': 0.9443901181221008, 'start': 109, 'end': 117, 'answer': 'Freeware'}
follow question : Microsoft live movie maker is an example of free software used for what purpose?
short answer : ['Video editing software']
{'score': 0.28220003843307495, 'start': 109, 'end': 117, 'answer': 'Freeware'}
{'rougeLsum': 50.90909090909091, 'length': 20.0, 'str_em': 50.0, 'Disambig-F1': 50.0}


rougeLsum      34.010879
length         24.550000
str_em         52.916667
Disambig-F1    49.041667
dtype: float64